In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# 1. Configure 4-bit quantization for 4 GB VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 2. Load Model and Processor
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# 3. Format Multimodal Input (Image + Text)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

# 4. Process Vision Info & Tokenize
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

# 5. Generate Response
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("--- Model Output ---")
print(output_text[0])

# 6. Measure Peak VRAM Usage for Acceptance Criteria 2
vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
print(f"\nPeak Allocated VRAM: {vram_gb:.2f} GB")

d:\university\Resesarch\qwen_vl_task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 824/824 [00:23<00:00, 35.14it/s] 


--- Model Output ---
The image depicts a small, fluffy animal, likely a Pallas's cat (also known as a Manul), walking on a snowy ground. The animal has a thick, woolly coat that is predominantly brown with darker stripes and spots. Its fur appears to be well-insulated for cold weather, which is typical of the species native to the cold regions of Central Asia.

The background features a snowy landscape with birch trees, which are common in the region where Pallas's cats are found. The trees have a white, snow-covered bark, indicating that it is winter. The snow on the ground is fluffy and untouched except for the

Peak Allocated VRAM: 2.60 GB


In [2]:
import gc

# Delete model and inputs from memory
del model
del processor
del inputs

# Force garbage collection and clear PyTorch CUDA cache
gc.collect()
torch.cuda.empty_cache()

print(f"Current VRAM Allocated: {torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB")

Current VRAM Allocated: 0.01 GB
